# 00 — What an SLM Actually Is

**Master syllabus coverage:** V2 0.1–0.2

## Why this matters

A clean component map prevents architecture, weights, tokenizer, runtime, and product behavior from becoming one vague idea. This distinction will guide the Python brain and future C++ engine boundary.

## Learning objectives

- Name the five major layers of a language-model product.
- Explain training and inference as two uses of next-token probabilities.
- Implement greedy and stochastic autoregressive generation.
- Measure a sequence with negative log-likelihood and perplexity.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## 1. The five layers people casually call “the model”

| Layer | What it is | Example artifact |
| --- | --- | --- |
| Architecture | The computation graph and tensor shapes | decoder-only Transformer |
| Parameters | Numbers learned by optimization | embedding and projection weights |
| Tokenizer | Reversible mapping between text and token IDs | vocabulary + merge rules |
| Runtime | Code that executes the graph and samples tokens | PyTorch now, C++ later |
| Application | Prompts, controls, safety, UI, and monitoring | Humor Machine |

Training repeatedly asks: “Given all earlier tokens, what probability should the
next token receive?” Inference repeatedly samples from that distribution.

$$p(x_{1:T}) = \prod_{t=1}^{T} p(x_t \mid x_{<t})$$

An SLM is “small” relative to frontier models, but it uses the same probabilistic
interface: context in, next-token logits out.


In [ ]:
import numpy as np

rng = np.random.default_rng(42)
vocabulary = ["<BOS>", "why", "did", "byte", "cross", "road", "?", "<EOS>"]
token_to_id = {token: index for index, token in enumerate(vocabulary)}
id_to_token = dict(enumerate(vocabulary))

print(token_to_id)
assert len(token_to_id) == len(id_to_token)


## 2. A model without a neural network

The table below is a tiny architecture whose “parameters” are handwritten
probabilities. Row $i$ represents the current token; column $j$ represents the
possible next token. Every row is a categorical distribution and must sum to one.

Shape: `transition[current_token, next_token] = [V, V]`, where $V=8$.


In [ ]:
V = len(vocabulary)
transition = np.zeros((V, V), dtype=np.float64)

def set_row(current: str, choices: dict[str, float]) -> None:
    for following, probability in choices.items():
        transition[token_to_id[current], token_to_id[following]] = probability

set_row("<BOS>", {"why": 1.0})
set_row("why", {"did": 1.0})
set_row("did", {"byte": 1.0})
set_row("byte", {"cross": 0.75, "?": 0.25})
set_row("cross", {"road": 1.0})
set_row("road", {"?": 1.0})
set_row("?", {"<EOS>": 1.0})
set_row("<EOS>", {"<EOS>": 1.0})

np.testing.assert_allclose(transition.sum(axis=1), 1.0)
print("parameter count:", transition.size)


## 3. Logits, temperature, sampling, and autoregression

Neural networks normally emit unrestricted **logits**, not probabilities. Softmax
turns logits into probabilities. Temperature changes distribution sharpness:

$$p_i = \frac{\exp(z_i / \tau)}{\sum_j \exp(z_j / \tau)}$$

Greedy decoding picks the largest value. Sampling draws from the distribution and
can produce different valid continuations. Autoregression appends one selected token
and feeds the longer context back into the model.


In [ ]:
def choose_next(probabilities: np.ndarray, temperature: float = 1.0) -> int:
    if temperature <= 0:
        return int(np.argmax(probabilities))
    safe = np.clip(probabilities, 1e-12, None)
    logits = np.log(safe) / temperature
    scaled = np.exp(logits - logits.max())
    scaled /= scaled.sum()
    return int(rng.choice(len(scaled), p=scaled))

def generate(temperature: float = 0.0, max_new_tokens: int = 12) -> list[str]:
    tokens = ["<BOS>"]
    for _ in range(max_new_tokens):
        row = transition[token_to_id[tokens[-1]]]
        next_token = id_to_token[choose_next(row, temperature)]
        tokens.append(next_token)
        if next_token == "<EOS>":
            break
    return tokens

greedy = generate(temperature=0.0)
sampled = [generate(temperature=0.8) for _ in range(4)]
print("greedy:", greedy)
print("sampled:", sampled)
assert greedy[-1] == "<EOS>"


## 4. Training versus inference

During **training**, the correct next token is known. We measure its negative log
probability, backpropagate, and update parameters. During **inference**, labels are
absent; the runtime repeatedly selects from predicted probabilities. The tokenizer
is required in both directions.

```text
text → tokenizer → IDs → architecture + weights → logits → sampler → next ID
     ↑                                                        │
     └────────────────────── decode IDs ←──────────────────────┘
```

Later lessons replace the table with learned n-grams, an MLP, and finally a decoder
Transformer. The interface remains next-token prediction.


In [ ]:
target_sequence = ["<BOS>", "why", "did", "byte", "cross", "road", "?", "<EOS>"]
losses = []
for current, target in zip(target_sequence, target_sequence[1:]):
    probability = transition[token_to_id[current], token_to_id[target]]
    losses.append(-np.log(probability))

mean_nll = float(np.mean(losses))
perplexity = float(np.exp(mean_nll))
print(f"mean NLL={mean_nll:.4f}, perplexity={perplexity:.4f}")
assert perplexity >= 1.0


## Failure modes to recognize

- **Probabilities do not sum to one:** sampling is no longer a valid categorical draw.
- **No stop condition:** autoregressive generation continues until an arbitrary length limit.
- **Tokenizer/model mismatch:** IDs refer to different tokens than the weights expect.
- **Calling everything the model:** ownership and debugging boundaries disappear.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Add an alternate continuation after `byte`, then compare greedy output with 20 sampled outputs.
2. Change temperature to 0.25, 1.0, and 2.0; describe diversity without using the word random.
3. Give one concrete file or data structure that will eventually represent each of the five layers.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can trace text to token IDs, logits/probabilities, a sampled ID, and decoded text while naming which component owns every step.

**Next:** 01 — CPU, GPU, and accelerator mental models.
